# Natural Language Processing Project - Menganalisis Dampak Terhadap Nilai Tukar Dolar
Arranged by:
*   Anders Emmanuel Tan (24/541351/PA/22964)
*   Azhar Maulana (24/533487/PA/22582)
*   Evan Razzan Adytaputra (24/545257/PA/23166)
*   Kukuh Agus Hermawan (24/533395/PA/22573)

In [6]:
%pip install -q trafilatura googlenewsdecoder beautifulsoup4 pandas requests

Note: you may need to restart the kernel to use updated packages.


In [7]:
# Cell 2: Configuration and noise filtering strategy
import os
import time
import requests
import pandas as pd
import xml.etree.ElementTree as ET
from datetime import datetime, timedelta
import trafilatura
from googlenewsdecoder import gnewsdecoder
from bs4 import BeautifulSoup

# Toggle TEST_MODE: False scrapes the full 5-year project horizon (Sept 2021 - Sept 2026)[cite: 1]
TEST_MODE = False

START_DATE = datetime(2021, 9, 1)
END_DATE = datetime(2026, 9, 1)

SAMPLE_INTERVAL_DAYS = 14     # Bi-weekly intervals for uniform temporal distribution[cite: 1]
WINDOW_DAYS = 4               # 4-day observation window per interval
MAX_PER_INTERVAL = 5 if TEST_MODE else 20

OUTPUT_DIR = "data/raw"
OUTPUT_CSV = "sample_geopolitical_news_filtered.csv" if TEST_MODE else "geopolitical_news.csv"
SAVE_PATH = os.path.join(OUTPUT_DIR, OUTPUT_CSV)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Strategic Filtering: Drop consumer finance, crypto, entertainment, and lifestyle[cite: 1]
EXCLUDE_KEYWORDS = [
    "nft", "meme", "doge", "crypto", "bitcoin", "ethereum", "token", 
    "dating", "wedding", "mansion", "celebrity", "movie", "box office",
    "tips for", "how to beat", "how to save", "credit score", "mortgage rate",
    "retirement", "best stocks", "undervalued"
]

# Strategic Filtering: Require sovereign macroeconomic and geopolitical signals[cite: 1]
CORE_GEOPOLITICAL_KEYWORDS = [
    "sanction", "sanctions", "tariff", "tariffs", "trade war", "embargo",
    "federal reserve", "central bank", "bank indonesia", "monetary policy",
    "rate hike", "rate cut", "interest rate", "foreign exchange", "forex", 
    "currency", "dollar index", "usd/idr", "treasury yield", "geopolitics", 
    "geopolitical", "foreign reserves", "sovereign debt", "war", "military"
]

print(f"Configuration set. Target file: {SAVE_PATH}")

Configuration set. Target file: data/raw\geopolitical_news.csv


In [8]:
# Cell 3: Helper routines for parsing and content filtering
session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9"
})

def parse_body(html_text: str) -> str:
    """Extracts readable article content with fallback to standard paragraph tags."""
    text = trafilatura.extract(html_text, include_comments=False, include_tables=False)
    if text and len(text.strip()) > 150:
        return text.strip()
    soup = BeautifulSoup(html_text, "html.parser")
    paragraphs = [p.get_text().strip() for p in soup.find_all("p") if len(p.get_text().strip()) > 40]
    extracted = " ".join(paragraphs)
    return extracted.strip() if len(extracted) > 150 else ""

def passes_strategic_filter(title: str, content: str) -> bool:
    """Applies negative noise suppression and positive macroeconomic relevance gating[cite: 1]."""
    title_lower = title.lower()
    content_lower = content.lower()
    full_text = title_lower + " " + content_lower

    # 1. Negative Noise Filtering[cite: 1]
    for bad_word in EXCLUDE_KEYWORDS:
        if bad_word in title_lower:
            return False

    # 2. Positive Filtering on title and lead text[cite: 1]
    lead_text = full_text[:1200]
    matched_signals = sum(1 for signal in CORE_GEOPOLITICAL_KEYWORDS if signal in lead_text)
    return matched_signals >= 1

In [9]:
# Cell 4: Check for previous CSV and identify completed checkpoints
checkpoints = []
curr = START_DATE
while curr <= END_DATE:
    checkpoints.append(curr)
    curr += timedelta(days=SAMPLE_INTERVAL_DAYS)

# Load past CSV data if it exists
if os.path.exists(SAVE_PATH):
    df_existing = pd.read_csv(SAVE_PATH)
    processed_titles = set(df_existing["title"].dropna().tolist())
    records = df_existing.to_dict("records")
    
    # Extract dates that have already been collected
    if not df_existing.empty and "published_at" in df_existing.columns:
        df_existing["published_at"] = pd.to_datetime(df_existing["published_at"])
        # A checkpoint is considered finished if we already have records from its time window
        completed_dates = {d.strftime("%Y-%m-%d") for d in df_existing["published_at"].dt.date}
    else:
        completed_dates = set()
        
    print(f"Resuming previous progress: Loaded {len(records)} articles from {SAVE_PATH}.")
else:
    processed_titles = set()
    records = []
    completed_dates = set()
    print("No previous progress found. Starting from scratch.")

print(f"Total target checkpoints: {len(checkpoints)}")

No previous progress found. Starting from scratch.
Total target checkpoints: 131


In [10]:
# Cell 5: Multi-threaded scraping with instant saving & checkpoint skipping
from concurrent.futures import ThreadPoolExecutor, as_completed

def process_single_item(item, dt_start):
    """Processes, decodes, and parses a single news item."""
    title_full = item.find("title").text if item.find("title") is not None else ""
    rss_link = item.find("link").text if item.find("link") is not None else ""
    pub_date = item.find("pubDate").text if item.find("pubDate") is not None else ""
    source_elem = item.find("source")
    source_name = source_elem.text if source_elem is not None else "Unknown"
    title_clean = title_full.rsplit(" - ", 1)[0] if " - " in title_full else title_full

    if not title_clean or title_clean in processed_titles:
        return None
    if any(bad in title_clean.lower() for bad in EXCLUDE_KEYWORDS):
        return None

    try:
        dt_parsed = pd.to_datetime(pub_date)
    except Exception:
        dt_parsed = dt_start

    # Resolve redirected Google News URL
    actual_url = rss_link
    try:
        decoded = gnewsdecoder(rss_link, interval=0.1)
        if isinstance(decoded, dict) and decoded.get("status"):
            actual_url = decoded.get("decoded_url", rss_link)
    except Exception:
        pass

    # Extract clean article body
    content_text = ""
    try:
        art_resp = session.get(actual_url, timeout=8)
        if art_resp.status_code == 200:
            content_text = parse_body(art_resp.text)
    except Exception:
        pass

    # Validate article content
    if len(content_text) > 200 and passes_strategic_filter(title_clean, content_text):
        return {
            "published_at": dt_parsed,
            "title": title_clean,
            "content": content_text,
            "source_domain": source_name,
            "language": "en",
            "url": actual_url
        }
    return None


for idx, dt_start in enumerate(checkpoints, 1):
    dt_end = dt_start + timedelta(days=WINDOW_DAYS)
    after_str = dt_start.strftime("%Y-%m-%d")
    before_str = dt_end.strftime("%Y-%m-%d")

    # Skip this checkpoint if we already have articles from this date range
    if after_str in completed_dates:
        print(f"[{idx}/{len(checkpoints)}] {after_str}: Already scraped. Skipping...")
        continue

    query = (
        f'(dollar OR USD OR "Federal Reserve" OR tariff OR sanctions OR inflation OR war) '
        f'(site:cnbc.com OR site:apnews.com) '
        f'after:{after_str} before:{before_str}'
    )
    rss_url = f"https://news.google.com/rss/search?q={requests.utils.quote(query)}&hl=en-US&gl=US&ceid=US:en"

    try:
        resp = session.get(rss_url, timeout=12)
        if resp.status_code == 200:
            root = ET.fromstring(resp.content)
            items = root.findall(".//item")[:MAX_PER_INTERVAL]

            new_articles = []
            # Concurrently fetch articles with 16 workers
            with ThreadPoolExecutor(max_workers=16) as executor:
                futures = [executor.submit(process_single_item, it, dt_start) for it in items]
                for future in as_completed(futures):
                    result = future.result()
                    if result and result["title"] not in processed_titles:
                        new_articles.append(result)
                        processed_titles.add(result["title"])

            if new_articles:
                records.extend(new_articles)
                completed_dates.add(after_str)

            # SAVE IMMEDIATELY to disk on every checkpoint iteration
            pd.DataFrame(records).to_csv(SAVE_PATH, index=False)

            print(f"[{idx}/{len(checkpoints)}] {after_str} to {before_str}: Saved {len(new_articles)} new articles (Total in CSV: {len(records)})")
        else:
            print(f"[{idx}/{len(checkpoints)}] RSS Error: HTTP {resp.status_code}")

    except Exception as e:
        print(f"[{idx}/{len(checkpoints)}] Checkpoint Error: {e}")

    time.sleep(0.4)

print("\nAll checkpoints processed!")

[1/131] 2021-09-01 to 2021-09-05: Saved 5 new articles (Total in CSV: 5)
[2/131] 2021-09-15 to 2021-09-19: Saved 7 new articles (Total in CSV: 12)
[3/131] 2021-09-29 to 2021-10-03: Saved 6 new articles (Total in CSV: 18)
[4/131] 2021-10-13 to 2021-10-17: Saved 6 new articles (Total in CSV: 24)
[5/131] 2021-10-27 to 2021-10-31: Saved 6 new articles (Total in CSV: 30)
[6/131] 2021-11-10 to 2021-11-14: Saved 10 new articles (Total in CSV: 40)
[7/131] 2021-11-24 to 2021-11-28: Saved 4 new articles (Total in CSV: 44)
[8/131] 2021-12-08 to 2021-12-12: Saved 6 new articles (Total in CSV: 50)
[9/131] 2021-12-22 to 2021-12-26: Saved 4 new articles (Total in CSV: 54)
[10/131] 2022-01-05 to 2022-01-09: Saved 6 new articles (Total in CSV: 60)
[11/131] 2022-01-19 to 2022-01-23: Saved 8 new articles (Total in CSV: 68)
[12/131] 2022-02-02 to 2022-02-06: Saved 14 new articles (Total in CSV: 82)
[13/131] 2022-02-16 to 2022-02-20: Saved 4 new articles (Total in CSV: 86)
[14/131] 2022-03-02 to 2022-03-06

KeyboardInterrupt: 

In [11]:
# Quick Diagnostics for the stalled 2024 date
import requests
import xml.etree.ElementTree as ET
from googlenewsdecoder import gnewsdecoder
import trafilatura

test_q = '(dollar OR USD OR "Federal Reserve") (site:cnbc.com OR site:apnews.com) after:2024-05-08 before:2024-05-12'
url = f"https://news.google.com/rss/search?q={requests.utils.quote(test_q)}&hl=en-US&gl=US&ceid=US:en"
resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)

print("RSS Status Code:", resp.status_code)
root = ET.fromstring(resp.content)
items = root.findall(".//item")
print(f"Items returned by Google RSS: {len(items)}")

if items:
    sample_link = items[0].find("link").text
    print("\nOriginal RSS link:", sample_link)
    try:
        decoded = gnewsdecoder(sample_link, interval=0.2)
        print("Decoded URL:", decoded)
        if isinstance(decoded, dict) and decoded.get("status"):
            real_url = decoded.get("decoded_url")
            html_resp = requests.get(real_url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
            print("Target Site Response Status:", html_resp.status_code)
            body = trafilatura.extract(html_resp.text)
            print("Extracted body length:", len(body) if body else 0)
    except Exception as e:
        print("Error during decode/fetch:", e)

RSS Status Code: 200
Items returned by Google RSS: 48

Original RSS link: https://news.google.com/rss/articles/CBMijgFBVV95cUxPSkVFanNwdzMzby1IZ1BnTlh0VkZWNl9kUDJiQVdiWW0tUFZPMzEweEhueGJUTjhGaWRCQWxpRkNXMjVFY281bjhNMEtqaWNxS2xLSEVadVlxR3poVzVfQzlrSUNPdDRJWVBjNUMxa2dONWIxb1AxX3RScWdmQV9rNFhCMndEd2E3SzhNQU9R?oc=5
Decoded URL: {'status': False, 'message': 'Request error in decode_url: 429 Client Error: Too Many Requests for url: https://www.google.com/sorry/index?continue=https://news.google.com/_/DotsSplashUi/data/batchexecute&q=EhAqCbrFOisl1wAAAAADxQAQGKicitUGIjAhPwmswufCnFDX3zu0O1updjUIusWDoun5WSmsd-HzeLxTJ9WQ4nk1Frcm81bi1RAyAnJSWgFD'}


In [ ]:
# Cell 6: Export finalized CSV and inspect dataset summary[cite: 1]
df_final = pd.DataFrame(records)

if not df_final.empty:
    df_final = df_final.dropna(subset=["title", "content", "published_at"]).drop_duplicates(subset=["title"])
    df_final["published_at"] = pd.to_datetime(df_final["published_at"])
    df_final = df_final.sort_values(by="published_at").reset_index(drop=True)
    df_final.to_csv(SAVE_PATH, index=False)
    
    print("=" * 60)
    print(f"Dataset compiled! Saved {len(df_final)} verified articles to: {SAVE_PATH}[cite: 1]")
    print("=" * 60)
    
    preview = df_final[["published_at", "title", "source_domain"]].copy()
    preview["content_chars"] = df_final["content"].str.len()
    display(preview.head(10))
else:
    print("No records saved. Check network connection or query parameters.")